# Gabarito — Lista 04 · SQL

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Exercícios do **módulo 04 · SQL**. São **14 exercícios**: 4 de teoria e 10 de código.
Tempo estimado: **2h30**.

As questões de teoria valem tanto quanto as de código. Boa parte dos erros de SQL não é de
sintaxe — é de entender mal o que a consulta faz com as linhas.

> **Use este arquivo depois de tentar.** Ler a solução e seguir em frente não fixa nada;
> o que fixa é travar, tentar, consultar e então refazer sem olhar.

### Antes de começar — se você está no Google Colab

Este notebook lê o banco de dados da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "05_Exercicios"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 140)

conexao = sqlite3.connect("../data/capacitacao.db")


def consultar(sql, parametros=None):
    return pd.read_sql_query(sql, conexao, params=parametros)


print("Conectado. Tabelas disponíveis:")
consultar("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")

## Parte I — Teoria

### Exercício 1

Explique, em duas ou três frases, a diferença entre `WHERE` e `HAVING`. Dê um exemplo de
pergunta que **só** pode ser respondida com `HAVING`.

**Comentário.** **`WHERE` filtra linhas antes do agrupamento; `HAVING` filtra grupos depois da
agregação.** A diferença decorre da ordem de execução: quando o `WHERE` roda, o `GROUP BY`
ainda não aconteceu e não existe grupo nenhum para contar — por isso o `WHERE` não consegue
enxergar `COUNT(*)`, `AVG(...)` ou qualquer outra agregação.

Pergunta que só o `HAVING` responde: *"quais setores têm mais de uma empresa?"* — o critério
é o resultado de uma contagem, que só existe depois de agrupar.

Quando a mesma restrição pode ser escrita nos dois lugares, prefira o `WHERE`: ele descarta
linhas antes do trabalho de agrupar, e em tabela grande a diferença de desempenho é
enorme.

### Exercício 2

Por que `SELECT COUNT(*)` e `SELECT COUNT(idade)` dão números diferentes na tabela
`clientes`? Qual dos dois você usaria para calcular a idade média, e por quê?

**Comentário.** `COUNT(*)` conta **linhas**; `COUNT(idade)` conta **valores não vazios** da coluna. Na
tabela `clientes` são 400 linhas e 379 idades preenchidas — os 21 vazios entram no primeiro
número e não no segundo.

Para a média, o certo é usar `AVG(idade)`, que já ignora os `NULL` nos dois lados da conta
(soma e divisor). Calcular `SUM(idade) / COUNT(*)` dividiria a soma de 379 valores por 400 e
subestimaria a média.

Mas há uma decisão analítica escondida aí: ignorar os vazios equivale a supor que os
clientes sem idade registrada se parecem com os demais. Nem sempre é verdade — às vezes o
dado falta justamente porque o caso é diferente.

### Exercício 3

Um colega escreveu a consulta abaixo e reclama que o total de aportes "deu muito mais alto
do que deveria".

```sql
SELECT SUM(c.aporte_mensal) AS total
FROM clientes c
INNER JOIN metas m ON c.perfil_investidor = m.perfil
```

Qual é a hipótese mais provável, e como você a testaria em uma consulta?

**Comentário.** A hipótese mais provável é que a chave `perfil` **se repete** na tabela `metas`. Quando
isso acontece, cada cliente casa com várias linhas de metas, o `JOIN` multiplica as linhas
e o `SUM` conta o mesmo aporte mais de uma vez.

O sintoma é traiçoeiro porque o resultado continua sendo um número plausível — não há erro,
não há aviso.

O teste:

```sql
SELECT perfil, COUNT(*) AS vezes
FROM metas
GROUP BY perfil
HAVING COUNT(*) > 1
```

Se isso devolver alguma linha, está confirmado. A conferência geral é contar antes e depois:

```sql
SELECT
    (SELECT COUNT(*) FROM clientes) AS antes,
    (SELECT COUNT(*) FROM clientes c
       INNER JOIN metas m ON c.perfil_investidor = m.perfil) AS depois
```

Se `depois` > `antes`, o `JOIN` multiplicou.

### Exercício 4

Qual a diferença entre `GROUP BY` e uma função de janela (`OVER`)? Dê um exemplo de
pergunta que só a janela resolve.

**Comentário.** **`GROUP BY` colapsa as linhas: uma linha por grupo. A função de janela preserva todas
as linhas** e apenas acrescenta uma coluna calculada sobre um conjunto de linhas vizinhas.

Pergunta que só a janela resolve: *"quais foram os 2 pregões de maior volume de **cada**
papel?"*. Com `GROUP BY` você obtém o maior (`MAX`), mas não os dois maiores — não há como.
Com janela, numera-se dentro de cada partição com `ROW_NUMBER()` e filtra-se por essa
numeração em uma CTE.

Outro exemplo: o retorno diário. Ele exige comparar cada linha com a **anterior**, o que
`GROUP BY` não sabe fazer, e `LAG()` resolve diretamente.

## Parte II — Código

### Exercício 5

Traga o `ticker`, a `empresa` e o `setor` de todas as empresas fundadas **antes de 1950**,
ordenadas da mais antiga para a mais recente.

In [ ]:
consultar("""
    SELECT ticker, empresa, setor, ano_fundacao
    FROM empresas
    WHERE ano_fundacao < 1950
    ORDER BY ano_fundacao
""")

**Comentário.** Filtro simples com `WHERE` e ordenação crescente por ano.

### Exercício 6

Quantos pregões a tabela `cotacoes` tem para cada papel? Traga `ticker` e a contagem,
do maior para o menor.

Depois responda em uma frase: o que esse resultado diz sobre o alinhamento das séries?

In [ ]:
consultar("""
    SELECT ticker, COUNT(*) AS pregoes
    FROM cotacoes
    GROUP BY ticker
    ORDER BY pregoes DESC
""")

**Comentário.** Todos os papéis têm exatamente 1.246 pregões. As séries estão perfeitamente alinhadas
no tempo: todo papel tem todo pregão, sem buracos. Isso é raro em base real e vale conferir
sempre — séries desalinhadas quebram comparações e produzem `NULL` inesperado em junções por
data.

### Exercício 7

Para cada **setor**, traga o número de empresas e o ano de fundação da mais antiga.
Mostre só os setores com **mais de uma** empresa.

In [ ]:
consultar("""
    SELECT
        setor,
        COUNT(*)          AS empresas,
        MIN(ano_fundacao) AS mais_antiga
    FROM empresas
    GROUP BY setor
    HAVING COUNT(*) > 1
    ORDER BY empresas DESC
""")

**Comentário.** O filtro é sobre o resultado de uma agregação, então é `HAVING`, não `WHERE`.

### Exercício 8

Classifique os clientes em faixas de patrimônio e conte quantos há em cada uma.

Atenção: `patrimonio_investido` está **sujo**. Os valores são texto em formato
brasileiro (ponto de milhar, vírgula decimal), uns com o prefixo `'R$ '` e outros sem, e há
vazios. **Antes de converter, conte quantos formatos existem** — a resposta muda a consulta.

Faixas sugeridas: até 50 mil, de 50 mil a 200 mil, acima de 200 mil, e "sem informação".

In [ ]:
consultar("""
    WITH convertido AS (
        SELECT
            id_cliente,
            CASE
                WHEN patrimonio_investido IS NULL THEN NULL
                ELSE CAST(REPLACE(REPLACE(REPLACE(patrimonio_investido, 'R$ ', ''),
                                          '.', ''), ',', '.') AS REAL)
            END AS patrimonio
        FROM clientes
    )
    SELECT
        CASE
            WHEN patrimonio IS NULL     THEN 'sem informação'
            WHEN patrimonio <= 50000    THEN 'até 50 mil'
            WHEN patrimonio <= 200000   THEN '50 a 200 mil'
            ELSE                             'acima de 200 mil'
        END AS faixa,
        COUNT(*) AS clientes
    FROM convertido
    GROUP BY faixa
    ORDER BY clientes DESC
""")

**Comentário.** Duas etapas em CTEs: primeiro converter o texto para número, depois classificar.

**A pegadinha está no formato.** A coluna tem dois: 91 valores com `'R$ '` na frente e 293
sem — todos em padrão brasileiro. Quem escreve um `CASE` que converte só os `R$` e manda o
resto para `CAST` direto destrói os outros 293: `CAST('412.666,21' AS REAL)` devolve
`412.666`, sem erro. Os três `REPLACE` aplicados a todos resolvem, porque o primeiro é
inofensivo em quem não tem prefixo.

A ordem dos `REPLACE` também importa — o ponto de milhar sai antes de a vírgula virar ponto
decimal. E a checagem de `NULL` vem **primeiro** no `CASE`, senão os vazios cairiam na
primeira faixa numérica.

### Exercício 9

Para cada setor, traga o volume total negociado em **2025** e o número de papéis do setor.

Use `JOIN`, e **confira** se a junção preservou o número de linhas.

In [ ]:
resultado = consultar("""
    SELECT
        e.setor,
        COUNT(DISTINCT e.ticker) AS papeis,
        SUM(c.volume)            AS volume_total
    FROM cotacoes c
    INNER JOIN empresas e ON c.ticker = e.ticker
    WHERE c.data >= '2025-01-01'
    GROUP BY e.setor
    ORDER BY volume_total DESC
""")

conferencia = consultar("""
    SELECT
        (SELECT COUNT(*) FROM cotacoes WHERE data >= '2025-01-01') AS antes,
        (SELECT COUNT(*)
           FROM cotacoes c
           INNER JOIN empresas e ON c.ticker = e.ticker
           WHERE c.data >= '2025-01-01')                           AS depois
""")
print(conferencia.to_string(index=False))
resultado

**Comentário.** Duas coisas para acertar aqui.

**`COUNT(DISTINCT ticker)`, não `COUNT(*)`.** Depois do `JOIN`, cada linha é um par
(papel, pregão) — um `COUNT(*)` contaria pregões, não papéis.

**A conferência.** A segunda consulta mostra que o `JOIN` não perdeu nem multiplicou:
o número de linhas de `cotacoes` em 2025 é o mesmo antes e depois.

### Exercício 10

Monte uma tabela com uma linha por setor e duas colunas de valor: quantas empresas são
**estatais** e quantas são **privadas**.

(Dica: agregação condicional — `SUM(CASE WHEN ...)`.)

In [ ]:
consultar("""
    SELECT
        setor,
        COUNT(*)                                              AS total,
        SUM(CASE WHEN controle = 'Estatal' THEN 1 ELSE 0 END) AS estatais,
        SUM(CASE WHEN controle = 'Privada' THEN 1 ELSE 0 END) AS privadas
    FROM empresas
    GROUP BY setor
    ORDER BY total DESC
""")

**Comentário.** O padrão da tabela cruzada: `CASE` devolve 1 quando a linha pertence à coluna e 0 quando
não, e `SUM` conta os uns. Com `SUM` contando ocorrências, o `ELSE 0` é necessário.

### Exercício 11

Traga, para a **VALE3**, o preço médio de fechamento ajustado de cada mês de 2025, ao lado
do IPCA e da Selic daquele mês.

Lembre: `cotacoes` é diária e `indicadores` é mensal. Agregue antes de juntar.

In [ ]:
consultar("""
    WITH mensal AS (
        SELECT
            strftime('%Y-%m', data)            AS ano_mes,
            ROUND(AVG(fechamento_ajustado), 2) AS preco_medio,
            COUNT(*)                           AS pregoes
        FROM cotacoes
        WHERE ticker = 'VALE3'
          AND data >= '2025-01-01'
        GROUP BY ano_mes
    )
    SELECT
        m.ano_mes,
        m.pregoes,
        m.preco_medio,
        i.ipca_mes_pct,
        i.selic_mes_pct
    FROM mensal m
    INNER JOIN indicadores i ON m.ano_mes = strftime('%Y-%m', i.data)
    ORDER BY m.ano_mes
""")

**Comentário.** A regra do módulo: **reduza a granularidade fina até a grossa, e só então junte.** Juntar
antes de agregar faria cada valor mensal ser repetido uma vez por pregão.

### Exercício 12

Calcule o **retorno diário** da ITUB4 em dezembro de 2025, usando `LAG`.

Traga a data, o fechamento anterior, o fechamento do dia e o retorno em %.

In [ ]:
consultar("""
    WITH com_anterior AS (
        SELECT
            data,
            ticker,
            fechamento_ajustado,
            LAG(fechamento_ajustado) OVER (
                PARTITION BY ticker ORDER BY data
            ) AS anterior
        FROM cotacoes
        WHERE ticker = 'ITUB4'
    )
    SELECT
        data,
        ROUND(anterior, 2)            AS ontem,
        ROUND(fechamento_ajustado, 2) AS hoje,
        ROUND((fechamento_ajustado / anterior - 1) * 100, 2) AS retorno_pct
    FROM com_anterior
    WHERE data >= '2025-12-01'
    ORDER BY data
""")

**Comentário.** Dois cuidados.

**`PARTITION BY ticker`**, mesmo filtrando um papel só: é o hábito que evita o bug quando a
consulta for reaproveitada para vários papéis.

**O filtro de data fica FORA da CTE.** Se ele entrasse junto da janela, o primeiro dia de
dezembro não teria "anterior" — a janela só enxerga o que sobrevive ao `WHERE`.

### Exercício 13

Para **cada papel**, traga os 3 pregões de maior amplitude (`maxima - minima`) de todo o
período. O resultado deve ter 24 linhas.

In [ ]:
consultar("""
    WITH ranqueado AS (
        SELECT
            ticker,
            data,
            ROUND(maxima - minima, 2) AS amplitude,
            ROW_NUMBER() OVER (
                PARTITION BY ticker ORDER BY (maxima - minima) DESC
            ) AS posicao
        FROM cotacoes
    )
    SELECT ticker, data, amplitude, posicao
    FROM ranqueado
    WHERE posicao <= 3
    ORDER BY ticker, posicao
""")

**Comentário.** O padrão "top N por grupo": `ROW_NUMBER()` particionado, dentro de uma CTE, e o filtro
pela posição por fora — funções de janela não podem ser usadas no `WHERE` da mesma
consulta.

### Exercício 14 — o ciclo completo

Escolha **uma** pergunta que exija juntar pelo menos duas tabelas e responda com o ciclo do
módulo 03:

1. escreva a pergunta em uma célula de texto;
2. responda com uma consulta SQL, usando CTEs para as etapas;
3. **confira** a junção (contou antes, contou depois);
4. traga o resultado para o pandas e faça um gráfico;
5. escreva a descoberta — inclusive se a resposta for "os dados não permitem concluir".

Sugestões, se preferir uma pronta: *o volume negociado se concentra em poucos papéis?* ·
*empresas mais antigas oscilam menos?* · *o retorno mensal dos papéis acompanha o dólar?*

In [ ]:
conferencia = consultar("""
    SELECT
        (SELECT COUNT(*) FROM cotacoes WHERE data >= '2025-01-01') AS antes,
        (SELECT COUNT(*) FROM cotacoes c
           INNER JOIN empresas e ON c.ticker = e.ticker
           WHERE c.data >= '2025-01-01')                           AS depois
""")
print(conferencia.to_string(index=False))

concentracao = consultar("""
    WITH volume_2025 AS (
        SELECT
            c.ticker,
            e.empresa,
            SUM(c.volume) AS volume_total
        FROM cotacoes c
        INNER JOIN empresas e ON c.ticker = e.ticker
        WHERE c.data >= '2025-01-01'
        GROUP BY c.ticker, e.empresa
    )
    SELECT
        ticker,
        empresa,
        volume_total,
        ROUND(100.0 * volume_total / SUM(volume_total) OVER (), 1) AS pct_do_total
    FROM volume_2025
    ORDER BY volume_total DESC
""")

concentracao["acumulado_pct"] = concentracao["pct_do_total"].cumsum()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(concentracao["ticker"], concentracao["pct_do_total"], color="#1f4e79")
ax.set_title("Participação de cada papel no volume negociado — 2025")
ax.set_ylabel("% do volume total")
for i, (pct, acum) in enumerate(zip(concentracao["pct_do_total"],
                                    concentracao["acumulado_pct"])):
    ax.text(i, pct + 0.5, f"{pct:.0f}%", ha="center", fontsize=9)
fig.tight_layout()
plt.show()

concentracao

**Comentário.** Solução possível — outras respostas igualmente válidas.

> **Pergunta:** o volume negociado se concentra em poucos papéis?

A conferência mostra que o `JOIN` preservou as 2.000 linhas de 2025. O gráfico mostra
concentração forte: os dois papéis mais líquidos respondem por boa parte do volume total.

**Descoberta.** O volume é bem mais concentrado que o número de papéis sugeriria — e a
limitação a declarar é óbvia: esta é uma amostra de 8 papéis escolhidos, não a B3 inteira.
A conclusão vale para a amostra e não pode ser estendida ao mercado.

---

## Sobre estas soluções

São **uma** possibilidade entre várias. Se a sua consulta é diferente e devolve o mesmo
resultado, ela também está certa — SQL costuma ter três ou quatro caminhos para a mesma
pergunta.

O critério é outro: você consegue explicar o que cada cláusula faz com as linhas, e
conferiu que a junção não perdeu nem multiplicou nada?

In [ ]:
conexao.close()
print("Conexão fechada.")